# 02 — Module B: Interest Rate Regression
Compare 8 regressors on predicting loan interest rate.

In [ ]:
import sys, os

# Add project root to path
PROJECT_ROOT = "/Users/nando/PycharmProjects/PythonProject/SmartLender"
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow

from src.data.loader import load_raw_data, split_data
from src.data.preprocessor import build_preprocessor, get_catboost_cat_indices
from src.models.registry import get_regressors
from src.models.trainer import run_arena
from src.evaluation.metrics import regression_metrics
from src.evaluation.comparison import save_comparison
from src.config import *

%matplotlib inline
mlflow.set_experiment('SmartLend')

## Load and Split Data

In [ ]:
df = load_raw_data(sample_size=200000)
# Drop rows where interest rate is missing
df = df.dropna(subset=[TARGET_INTEREST_RATE])
X_train, X_test, y_train, y_test = split_data(df, TARGET_INTEREST_RATE)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Interest rate range: {y_train.min():.2f}% to {y_train.max():.2f}%")

## Build Preprocessor

In [ ]:
preprocessor = build_preprocessor(X_train)
cat_indices = get_catboost_cat_indices(X_train)

## Run the Arena — All 8 Regressors

In [ ]:
regressors = get_regressors()
comparison_df, results = run_arena(
    models=regressors,
    preprocessor=preprocessor,
    X_train=X_train, y_train=y_train,
    X_test=X_test, y_test=y_test,
    metrics_fn=regression_metrics,
    cat_feature_indices=cat_indices,
    module_name='module_b',
)

comparison_df = comparison_df.sort_values('r2', ascending=False)
print(comparison_df.to_string(index=False))

## Save Comparison

In [ ]:
save_comparison(comparison_df, 'module_b')

## Actual vs Predicted (Top 3 Models)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
top_3 = comparison_df.head(3)['algorithm'].tolist()

for i, name in enumerate(top_3):
    model = [r for r in results if r['algorithm'] == name][0]['model']
    if 'CatBoost' in name:
        X_test_pred = X_test.copy()
        for col in X_test_pred.select_dtypes(include=['object', 'category']).columns:
            X_test_pred[col] = X_test_pred[col].fillna('Missing').astype(str)
        y_pred = model.predict(X_test_pred)
    else:
        y_pred = model.predict(X_test)

    axes[i].scatter(y_test, y_pred, alpha=0.1, s=1, color='#3498db')
    axes[i].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
    axes[i].set_xlabel('Actual Interest Rate (%)')
    axes[i].set_ylabel('Predicted Interest Rate (%)')
    axes[i].set_title(name)

plt.suptitle('Actual vs Predicted — Module B', fontsize=16)
plt.tight_layout()
plt.savefig('results/figures/module_b_actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()

## Feature Importance Comparison

In [ ]:
import plotly.express as px

# Compare feature importances from tree-based models
importance_data = []
for result in results:
    name = result['algorithm']
    model = result['model']
    # Skip Linear Regression (no feature_importances_)
    if name == 'Linear Regression':
        continue
    try:
        if 'CatBoost' in name:
            importances = model.feature_importances_
            feat_names = list(X_train.columns)
        else:
            inner_model = model.named_steps['model']
            importances = inner_model.feature_importances_
            feat_names = list(preprocessor.get_feature_names_out())

        for fn, imp in zip(feat_names, importances):
            importance_data.append({'algorithm': name, 'feature': fn, 'importance': imp})
    except Exception:
        pass

if importance_data:
    imp_df = pd.DataFrame(importance_data)
    # Top 10 features per model
    top_feats = imp_df.groupby('algorithm').apply(
        lambda x: x.nlargest(10, 'importance')
    ).reset_index(drop=True)

    fig = px.bar(top_feats, x='importance', y='feature', color='algorithm',
                 facet_col='algorithm', facet_col_wrap=4,
                 title='Top 10 Feature Importances by Algorithm')
    fig.update_layout(height=800, showlegend=False)
    fig.show()